In [1]:
import os
import io
import time
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image

from datasets import load_dataset

from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    RandomHorizontalFlip,
    RandomRotation,
    ColorJitter,
    Resize,
    CenterCrop,
    ToTensor,
    Normalize,
)

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from tqdm.auto import tqdm

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
dataset = load_dataset("AI-Lab-Makerere/beans")

print(dataset)
print(dataset["train"].features)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  144MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.5MB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})
{'image_file_path': Value('string'), 'image': Image(mode=None, decode=True), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}


In [4]:
label_feature = dataset["train"].features["labels"]
class_names = label_feature.names

id2label = {
    index: label
    for index, label in enumerate(class_names)
}

label2id = {
    label: index
    for index, label in id2label.items()
}

print("Classes:", class_names)
print("id2label:", id2label)

Classes: ['angular_leaf_spot', 'bean_rust', 'healthy']
id2label: {0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'}


In [ ]:
MODEL_ID = "facebook/convnext-tiny-224"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
imagenet_model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID
).to(device)

imagenet_model.eval()

print("Model loaded:", MODEL_ID)
print("Original classes:", imagenet_model.config.num_labels)

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  114MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

Model loaded: facebook/convnext-tiny-224
Original classes: 1000


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            